In [ ]:
import os, sys, imageio
import gymnasium as gym
import numpy as np
import torch
import matplotlib

matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from pendulum_train import PendulumSafety  # custom wrapper for Pendulum-v1


# region: supporting functions
# Pendulum-v1: obs = [cos(theta), sin(theta), theta_dot]
def angle_from_obs(obs):
    c, s = obs[..., 0], obs[..., 1]
    return np.arctan2(s, c)  # [-pi, pi]


def adversarial_torque(theta, magnitude=1.0):
    # Push further away from 0 with fixed magnitude
    return magnitude * (1.0 if np.sign(theta) >= 0 else -1.0)


def set_upright(env):
    """Force the pendulum to upright state (theta=0, theta_dot=0)."""
    env.unwrapped.state = np.array([0.0, 0.0], dtype=np.float32)
    return np.array([1.0, 0.0, 0.0], dtype=np.float32)


def _mark_termination(frame: np.ndarray, thickness: int = 8, color=(255, 0, 0)) -> np.ndarray:
    """Add a red border to indicate termination on this frame."""
    h, w, _ = frame.shape
    t = thickness
    f = frame.copy()
    f[:t, :, :] = color  # top
    f[-t:, :, :] = color  # bottom
    f[:, :t, :] = color  # left
    f[:, -t:, :] = color  # right
    return f


def draw_arrow_on_frame(frame, obs, action, shielded, intervened, terminated):
    """
    Overlay an arrow on RGB frame indicating torque direction and whether shielding intervened.
    """
    theta = float(angle_from_obs(obs))
    action_scalar = float(np.asarray(action, dtype=np.float32).reshape(-1)[0])

    fig, ax = plt.subplots(figsize=(4, 4), dpi=72)
    ax.imshow(frame)
    ax.axis("off")

    cx, cy = float(frame.shape[1] / 2.0), float(frame.shape[0] / 2.0)

    torque_sign = 1.0 if action_scalar >= 0.0 else -1.0
    dx = float(40.0 * np.cos(theta + np.pi / 2.0) * torque_sign)
    dy = float(-40.0 * np.sin(theta + np.pi / 2.0) * torque_sign)

    if not shielded:
        color = "blue"
    elif intervened:
        color = "red"
    else:
        color = "green"

    ax.arrow(
        cx, cy, dx, dy, color=color, head_width=8.0, head_length=12.0, length_includes_head=True
    )

    fig.canvas.draw()
    plotted_frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    plotted_frame = plotted_frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)

    if terminated:
        plotted_frame = _mark_termination(plotted_frame, thickness=8, color=(255, 0, 0))

    return plotted_frame


# ---------- Safety filtering ----------
def state_safety_value(model, obs, action):
    device = getattr(model, "device", "cpu")
    obs_t = torch.as_tensor(obs, device=device, dtype=torch.float32)
    action_t = torch.as_tensor(action, device=device, dtype=torch.float32)

    # Ensure batch dimension
    if obs_t.ndim == 1:
        obs_t = obs_t.unsqueeze(0)
    if action_t.ndim == 1:
        action_t = action_t.unsqueeze(0)

    with torch.no_grad():
        v = model.critic(obs_t, action_t)

    # Handle case where critic returns (q1, q2)
    if isinstance(v, (tuple, list)):
        v = torch.min(v[0], v[1])

    return v.squeeze().cpu().numpy()


def sample_safe_initial_obs(
    env, model, threshold: float = 0.0, max_tries: int = 500, seed: int = 0
):
    """
    Randomly sample an initial state s0 with Q_s*(s0) > threshold.
    Tries different resets (and optionally a few random warmup steps).
    """
    rng = np.random.default_rng(seed)
    for _ in range(max_tries):
        obs, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
        action = 0 * env.action_space.sample()

        val = state_safety_value(model, obs, action)
        if val > threshold:
            return obs
    raise RuntimeError("Failed to sample a safe initial state with Q_s(s0) > threshold.")


def rollout(
    model, path, steps: int = 600, shield: bool = False, seed: int = 0, threshold: float = 0.0
):
    env = gym.make("Pendulum-v1", render_mode="rgb_array")
    env = PendulumSafety(env)  # ensure reward == g(s)
    env.action_space.seed(seed)

    # --- 1) random safe initial state
    obs = sample_safe_initial_obs(env, model, threshold=threshold, max_tries=1000, seed=seed)

    frames = []

    for _ in range(steps):
        # --- 2) base action: random sample
        task_action = env.action_space.sample()

        # --- 3) safe fallback action: query the model
        safe_action, _ = model.predict(obs, deterministic=False)

        # --- 4) decide action based on state safety value
        if shield:
            q_state = state_safety_value(model, obs, task_action)
            use_safe = (q_state < threshold)
            action = safe_action if use_safe else task_action
            intervened = bool(use_safe)
        else:
            action = task_action
            use_safe = False
            intervened = False

        obs, _g, terminated, truncated, _ = env.step(action)
        raw_frame = env.render()

        plotted_frame = draw_arrow_on_frame(
            raw_frame, obs, action, use_safe, intervened, terminated
        )
        frames.append(plotted_frame)

        if terminated or truncated:
            # Re-sample a new safe initial state after episode end
            obs = sample_safe_initial_obs(env, model, threshold=0.0, max_tries=1000, seed=seed)

    env.close()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    imageio.mimsave(path, frames, fps=30)


# endregion


### SafetySAC

In [2]:
from safety_sb3 import SafetySAC  # or: from safety_sac import SafetySAC

outdir = "pendulum_output"
os.makedirs(outdir, exist_ok=True)

# Recreate the env (needed for predict/rollouts)
env = PendulumSafety(gym.make("Pendulum-v1"))

# Load the trained model
MODEL_PATH = "models/pendulum"
model = SafetySAC.load(MODEL_PATH, env=env, device="auto")
model.policy.set_training_mode(False)  # ensure eval mode (no dropout/BN updates)

# Unshielded vs. Shielded GIFs
rollout(model, os.path.join(outdir, "test_unshielded.gif"), steps=100, shield=False, seed=42)
rollout(
    model, os.path.join(outdir, "test_shielded.gif"), steps=100, shield=True, seed=42, threshold=0.2
)


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


### SafetyDQN

In [ ]:
import os, imageio, numpy as np, torch as th
from safety_sb3 import SafetyDQN
import gymnasium as gym
from gymnasium import spaces
from pendulum_train import DiscretizeActionWrapper


# region: supporting functions
def rollout_disc(
    model,
    path,
    steps: int = 600,
    shield: bool = False,
    seed: int = 0,
    threshold: float = 0.0,
    n_bins: int = 11,
):
    """
    SafetyDQN rollout on Pendulum with discretized actions.
    - task action = random discrete action (simulates an unsafe controller)
    - safe action = argmax_a Q_s(s,a) from the SafetyDQN
    - if shield: intervene (choose safe action) when Q_s(s, a_task) < threshold

    Saves a GIF.
    """

    def _state_safety_value(model, obs, action_idx):
        """
        Returns Q_s(s, a_idx) from a DQN-style model (SafetyDQN) on a single obs.
        Uses model.policy.q_net, which outputs [batch, n_actions].
        """
        if isinstance(obs, np.ndarray):
            obs_t = th.as_tensor(obs, dtype=th.float32, device=model.device).unsqueeze(0)
        else:
            obs_t = obs.to(model.device).float().unsqueeze(0)

        with th.no_grad():
            q_all = model.policy.q_net(obs_t)  # shape [1, n_actions]
            q_val = q_all[0, int(action_idx)].item()
        return q_val

    def _q_sa(model, obs, action_idx):
        """
        Return Q_s(s, a_idx) from a SafetyDQN-style model on a single observation.
        Uses model.policy.q_net (SB3 DQN) under the hood.
        """
        # Ensure obs batch
        if isinstance(obs, np.ndarray):
            obs_t = th.as_tensor(obs, dtype=th.float32, device=model.device).unsqueeze(0)
        else:
            # assume torch tensor
            obs_t = obs.to(model.device).float().unsqueeze(0)

        with th.no_grad():
            # SB3 DQN: q_net outputs [batch, n_actions]
            q_all = model.policy.q_net(obs_t)
            q_value = q_all[0, int(action_idx)].item()
        return q_value

    def _safety_argmax_action(model, obs):
        """
        Argmax over Q_s(s, a) for discrete actions (SafetyDQN policy).
        """
        if isinstance(obs, np.ndarray):
            obs_t = th.as_tensor(obs, dtype=th.float32, device=model.device).unsqueeze(0)
        else:
            obs_t = obs.to(model.device).float().unsqueeze(0)
        with th.no_grad():
            q_all = model.policy.q_net(obs_t)  # [1, n_actions]
            act_idx = th.argmax(q_all, dim=1).item()
        return act_idx

    def _sample_safe_initial_obs(env, model, threshold=0.0, max_tries=1000, seed=0):
        """
        Keep resetting until there exists some discrete action with Q_s(s,a) >= threshold.
        Works with DiscretizeActionWrapper (env.action_space is Discrete).
        """
        rng = np.random.default_rng(seed)
        nA = env.action_space.n

        for _ in range(max_tries):
            obs, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
            # check the best action’s safety value
            a_best = _safety_argmax_action(model, obs)
            val = _state_safety_value(model, obs, a_best)
            if val >= threshold:
                return obs

        # if we didn’t find a "safe" state, just return the last one
        return obs

    # 1) Make rgb env + safety reward and discrete action wrapper
    base = gym.make("Pendulum-v1", render_mode="rgb_array")
    env = PendulumSafety(base)  # reward == g(s)
    env = DiscretizeActionWrapper(env, n_bins=n_bins)
    assert isinstance(env.action_space, spaces.Discrete)
    env.action_space.seed(seed)

    # 2) Random safe initial state (using your helper)
    obs = _sample_safe_initial_obs(env, model, threshold=threshold, max_tries=1000, seed=seed)

    frames = []

    for _ in range(steps):
        # --- task action: uniformly random discrete action
        task_action = int(env.action_space.sample())

        # --- safe fallback: argmax_a Q_s(s,a)
        safe_action = _safety_argmax_action(model, obs)

        # --- shield decision based on state-action safety value
        if shield:
            q_state = _q_sa(model, obs, task_action)
            use_safe = bool(q_state < threshold)
            action = safe_action if use_safe else task_action
            intervened = use_safe
        else:
            action = task_action
            use_safe = False
            intervened = False

        # Step with DISCRETE action; wrapper maps to Box for the base env
        obs, g_reward, terminated, truncated, _ = env.step(action)

        raw_frame = env.render()
        plotted = draw_arrow_on_frame(raw_frame, obs, action, use_safe, intervened, terminated)
        frames.append(plotted)

        if terminated or truncated:
            # Re-sample a new *safe* start
            obs = _sample_safe_initial_obs(
                env, model, threshold=threshold, max_tries=1000, seed=seed
            )

    env.close()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    imageio.mimsave(path, frames, fps=30)


# endregion

outdir = "pendulum_disc_output"
os.makedirs(outdir, exist_ok=True)

# Recreate the env (needed for predict/rollouts)
env = PendulumSafety(gym.make("Pendulum-v1"))
env = DiscretizeActionWrapper(env, n_bins=21)

# Load the trained model
MODEL_PATH = "models/pendulum_disc"
model = SafetyDQN.load(MODEL_PATH, env=env, device="auto")
model.policy.set_training_mode(False)  # ensure eval mode (no dropout/BN updates)

# Unshielded vs. Shielded GIFs
rollout_disc(
    model, os.path.join(outdir, "test_unshielded.gif"), steps=100, shield=False, seed=42, n_bins=21
)
rollout_disc(
    model, os.path.join(outdir, "test_shielded.gif"), steps=100, shield=True, seed=42,
    threshold=0.2, n_bins=21
)


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


AttributeError: 'SafetyDQN' object has no attribute 'critic'